# 01 · Quickstart: VB-GNN on simulated data
Simulate a non-linear additive-noise model on an Erdős–Rényi graph (main text Section 3.1), fit VB-GNN
(SI Algorithm 1), select edges with the posterior-FDR rule (SI Section 1.2) and score the recovered graph.

In [ ]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from vbgnn import VBGNN, VBGNNConfig, posterior_fdr_select, structure_metrics, edge_auroc
from vbgnn.simulation import make_graph, simulate_anm

rng = np.random.default_rng(0)
B = make_graph("ER-2", 15, rng)                   # B[i, j] = 1 means i -> j
X = simulate_anm(B, n=1000, rng=rng, rho=0.3)     # correlated noise, rho = 0.3
print(X.shape, "true edges:", int(B.sum()))

In [ ]:
model = VBGNN(VBGNNConfig(iters=3000, seed=0)).fit(X)
P = model.edge_probs                              # posterior edge probabilities pi_hat
selected, expected_fdp = posterior_fdr_select(P, alpha=0.05)
print("selected edges:", int(selected.sum()), " posterior E[FDP] = %.3f" % expected_fdp)
print(structure_metrics(B, selected), " AUROC = %.3f" % edge_auroc(B, P))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
for ax, M, t in zip(axes, [B, P], ["true graph", "posterior edge probabilities"]):
    im = ax.imshow(M, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(t, loc="left"); ax.set_xlabel("target j"); ax.set_ylabel("source i")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

In [ ]:
import pandas as pd
pd.DataFrame(model.history).set_index("iter")[["nll", "kl", "struct", "acyc", "D"]].tail()